In [ ]:
# ── Configuration and paths ──
import pandas as pd
import numpy as np
from pathlib import Path
from neuroCombat import neuroCombat

# Paths
BASE_DIR = Path(".")
input_dir = BASE_DIR / "outputs"
output_dir = BASE_DIR / "outputs"

# Input files
otu_path      = input_dir / "merged_otu_counts.csv"
meta_path     = input_dir / "merged_metadata.csv"

# Output files
clr_path      = output_dir / "preprocessed_otu_clr.csv"
delta_path    = output_dir / "delta_otu_clr.csv"
report_path   = output_dir / "preprocessing_report.txt"

print("Paths configured.")
print(f"  OTU counts : {otu_path}")
print(f"  Metadata   : {meta_path}")

In [ ]:
# ── Load OTU counts and metadata ──
otu = pd.read_csv(otu_path, index_col=0)

# Load metadata — rows = samples
meta = pd.read_csv(meta_path, index_col="sample_uid")

# Align: keep only samples present in both
common_samples = otu.columns.intersection(meta.index)
otu  = otu[common_samples]
meta = meta.loc[common_samples]

print(f"OTU table   : {otu.shape[0]:,} OTUs × {otu.shape[1]:,} samples")
print(f"Metadata    : {meta.shape[0]:,} samples × {meta.shape[1]:,} columns")
print(f"Common samples: {len(common_samples):,}")
print(f"\nMetadata columns: {list(meta.columns)}")

In [ ]:
# ── Cast OTU table to numeric ──
otu = otu.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float32)

print(f"OTU dtype: {otu.dtypes.unique()}")
print(f"Shape: {otu.shape}")


In [ ]:
# ── Per-study prevalence filter (5%, union) ──

studies = meta['study'].unique()
retained_otus = set()

for study in studies:
    study_samples = meta[meta['study'] == study].index
    otu_study = otu.loc[:, study_samples]
    n = otu_study.shape[1]
    min_s = int(np.ceil(0.05 * n))
    presence = (otu_study > 0).sum(axis=1)
    passing = presence[presence >= min_s].index.tolist()
    retained_otus.update(passing)
    print(f"{study:<25} {n:>4} samples, {min_s:>2} min, {len(passing):>4} OTUs passing")

print(f"\nTotal OTUs retained (union): {len(retained_otus):,}")

In [ ]:
# ── Apply per-study filter and compute CLR ──

otu_filtered = otu.loc[list(retained_otus)]
print(f"Filtered OTU matrix: {otu_filtered.shape[0]:,} OTUs × {otu_filtered.shape[1]:,} samples")

# Relative abundance
ra = otu_filtered.div(otu_filtered.sum(axis=0), axis=1)

# CLR
pseudocount = 1e-6
ra_pseudo = ra + pseudocount
clr = np.log(ra_pseudo).subtract(np.log(ra_pseudo).mean(axis=0), axis=1)

# Verify
print(f"CLR shape  : {clr.shape}")
print(f"Any NaN    : {clr.isna().any().any()}")
print(f"Any Inf    : {np.isinf(clr.values).any()}")
print(f"CLR mean (should be ~0): {clr.mean(axis=0).mean():.2e}")

In [ ]:
# ── Drop zero-sum samples and recompute CLR ──

sample_totals = otu_filtered.sum(axis=0)
zero_sum_samples = sample_totals[sample_totals == 0]
print(f"Zero-sum samples: {len(zero_sum_samples)}")

otu_filtered = otu_filtered.drop(columns=zero_sum_samples.index)
meta = meta.drop(index=zero_sum_samples.index)

# Recompute relative abundance + CLR
ra = otu_filtered.div(otu_filtered.sum(axis=0), axis=1)
pseudocount = 1e-6
ra_pseudo = ra + pseudocount
clr = np.log(ra_pseudo).subtract(np.log(ra_pseudo).mean(axis=0), axis=1)

print(f"OTU matrix : {otu_filtered.shape[0]:,} OTUs × {otu_filtered.shape[1]:,} samples")
print(f"Metadata   : {meta.shape[0]:,} samples")
print(f"Any NaN    : {clr.isna().any().any()}")
print(f"Any Inf    : {np.isinf(clr.values).any()}")
print(f"CLR mean   : {clr.mean(axis=0).mean():.2e}")

In [ ]:
# ── ComBat batch correction ──

# neuroCombat expects features x samples (OTUs x samples)
# Covariates dataframe must be samples x covariates

# Encode categorical covariates as integers
covars = meta[['study', 'treatment', 'timepoint']].copy()
covars['treatment'] = (covars['treatment'] == 'fiber').astype(int)
covars['timepoint'] = (covars['timepoint'] == 'after').astype(int)

print("Covariate value counts:")
print(covars['treatment'].value_counts().to_dict())
print(covars['timepoint'].value_counts().to_dict())
print(covars['study'].value_counts().to_dict())

# Run ComBat
combat_output = neuroCombat(
    dat=clr.values,          # numpy array, OTUs x samples
    covars=covars,
    batch_col='study',
    categorical_cols=['treatment', 'timepoint']
)

clr_corrected = pd.DataFrame(
    combat_output['data'],
    index=clr.index,
    columns=clr.columns
)

print(f"\nComBat complete.")
print(f"Corrected matrix shape: {clr_corrected.shape}")
print(f"Any NaN: {clr_corrected.isna().any().any()}")

In [ ]:
# ── Delta computation (after − before, mean-collapsed) ──

otu_cols = clr_corrected.index.tolist()

# Work from clr_t (samples x OTUs + metadata) — rebuild cleanly
clr_t = clr_corrected.T.copy()
clr_t.index.name = 'sample_uid'
clr_t = clr_t.join(meta[['study', 'subject_id', 'treatment', 'timepoint']])

# Group by (study, subject_id, treatment, timepoint) and average OTU values
grouped = clr_t.groupby(['study', 'subject_id', 'treatment', 'timepoint'])[otu_cols].mean()

# Split before and after
before = grouped.xs('before', level='timepoint')
after  = grouped.xs('after',  level='timepoint')

# Delta on matched pairs
common_idx = before.index.intersection(after.index)
delta = after.loc[common_idx] - before.loc[common_idx]

print(f"Paired subjects  : {len(common_idx):,}")
print(f"Delta matrix     : {delta.shape}")
print(f"Any NaN          : {delta.isna().any().any()}")

# Unpaired breakdown
unpaired_before = len(before.index.difference(after.index))
unpaired_after  = len(after.index.difference(before.index))
print(f"Unpaired before-only : {unpaired_before}")
print(f"Unpaired after-only  : {unpaired_after}")

In [ ]:
# ── Save outputs ──

# 1. Batch-corrected CLR matrix — transpose to samples x OTUs for usability
clr_out = clr_corrected.T  # samples x OTUs
clr_out.index.name = 'sample_uid'
clr_out.to_csv(output_dir / "preprocessed_otu_clr.csv")
print(f"Saved preprocessed_otu_clr.csv : {clr_out.shape}")

# 2. Delta matrix — already samples x OTUs, index is (study, subject_id, treatment)
delta.index.names = ['study', 'subject_id', 'treatment']
delta.to_csv(output_dir / "delta_otu_clr.csv")
print(f"Saved delta_otu_clr.csv        : {delta.shape}")

# 3. Updated metadata (3 zero-sum samples dropped)
meta.to_csv(output_dir / "merged_metadata_clean.csv")
print(f"Saved merged_metadata_clean.csv: {meta.shape}")


In [ ]:
# ── Preprocessing report ──
report_lines = [
    "=" * 60,
    "STAGE 2 PREPROCESSING REPORT",
    "=" * 60,
    "",
    "--- Input ---",
    f"OTU table        : 68,832 OTUs x 2,286 samples",
    f"Metadata         : 2,286 samples x 8 columns",
    "",
    "--- Prevalence Filtering ---",
    f"Strategy         : Per-study 5% prevalence, global union",
    f"Rationale        : Global 5% filter eliminated 4 studies entirely",
    f"OTUs retained    : 9,612 (from 68,832)",
    f"OTUs removed     : 59,220 (86.0%)",
    "",
    "--- Zero-sum Sample Removal ---",
    f"Samples removed  : 3 (Rasmussen x2, Tap x1)",
    f"Reason           : Zero counts across all 9,612 retained OTUs",
    f"Samples remaining: 2,283",
    "",
    "--- CLR Transformation ---",
    f"Method           : Relative abundance + pseudocount (1e-6) + CLR",
    f"Any NaN/Inf      : None",
    "",
    "--- ComBat Batch Correction ---",
    f"Package          : neuroCombat",
    f"Batch variable   : study (11 levels)",
    f"Protected vars   : treatment (fiber/control), timepoint (before/after)",
    f"Input shape      : 9,612 OTUs x 2,283 samples",
    f"Output shape     : 9,612 OTUs x 2,283 samples",
    "",
    "--- Delta Computation ---",
    f"Method           : Mean CLR(after) - Mean CLR(before) per subject",
    f"Pairing key      : (study, subject_id, treatment)",
    f"Note             : Multiple timepoints per subject collapsed by mean (Baxter)",
    f"Paired subjects  : 520",
    f"Unpaired before  : 12 (excluded from delta only)",
    f"Unpaired after   : 10 (excluded from delta only)",
    f"Delta shape      : 520 x 9,612",
    "",
    "--- Outputs ---",
    f"preprocessed_otu_clr.csv    : 2,283 samples x 9,612 OTUs (batch-corrected CLR)",
    f"delta_otu_clr.csv           : 520 subjects x 9,612 OTUs (after - before delta)",
    f"merged_metadata_clean.csv   : 2,283 samples x 8 columns",
    "",
    "=" * 60,
]

report_text = "\n".join(report_lines)
print(report_text)

with open(report_path, "w") as f:
    f.write(report_text)

print(f"\nReport saved to {report_path}")